# 🇺🇿 Notebook 1 — Data Exploration
**Project:** Uzbek Sentiment Analysis | **Author:** Asliddin | Presidential School, Namangan

**Series:** Asliddin Builds #03

---
In this notebook we:
1. Load and inspect the multi-domain Uzbek dataset (news / telegram / reviews)
2. Analyze class and domain distribution
3. Explore text length patterns
4. Visualize most common sentiment-bearing words
5. Build train/val/test split

In [ ]:
import sys
sys.path.append('../src')
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split

plt.rcParams.update({'figure.dpi':120, 'font.size':11})
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 1. Load Dataset

Data sources:
- **News**: kun.uz, daryo.uz, gazeta.uz (scraped headlines + comments)
- **Telegram**: Public Uzbek channels (Xabar, Daryo, UzReport)
- **Reviews**: OLX.uz, Uzum Market product reviews

Place CSVs in `../data/raw/` as `news.csv`, `telegram.csv`, `reviews.csv`

Each must have: `text`, `label` (negative/neutral/positive)

In [ ]:
raw_dir = '../data/raw'
dfs = {}
for domain, fname in [('news','news.csv'),('telegram','telegram.csv'),('review','reviews.csv')]:
    path = os.path.join(raw_dir, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['domain'] = domain
        dfs[domain] = df
        print(f'[{domain}] {len(df):,} samples')
    else:
        print(f'[{domain}] Not found — using demo placeholder')
        np.random.seed(42)
        dfs[domain] = pd.DataFrame({
            'text': [f'Namuna {domain} matni {i}' for i in range(200)],
            'label': np.random.choice(['negative','neutral','positive'],[200],p=[0.28,0.41,0.31]),
            'domain': domain
        })

df_all = pd.concat(dfs.values(), ignore_index=True)
df_all['label'] = df_all['label'].str.lower().str.strip()
df_all = df_all[df_all['label'].isin(['negative','neutral','positive'])]
print(f'\nTotal: {len(df_all):,} samples')

## 2. Class × Domain Distribution

In [ ]:
label_colors = {'negative':'#ef5350','neutral':'#ffd54f','positive':'#66bb6a'}
domain_names = {'news':'News','telegram':'Telegram','review':'Reviews'}

fig, axes = plt.subplots(1,3,figsize=(15,5))
fig.suptitle('Sentiment Distribution by Domain', fontsize=14, fontweight='bold')

for ax, (domain, df_d) in zip(axes, dfs.items()):
    counts = df_d['label'].value_counts()
    colors = [label_colors.get(l,'#888') for l in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(f'{domain_names[domain]} (n={len(df_d):,})', fontweight='bold')
    ax.set_ylabel('Count')
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, count+2, str(count),
                ha='center', fontweight='bold', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Neutral class is most common — we handle this with class-weighted loss.')

## 3. Text Length Distribution

In [ ]:
df_all['text_len'] = df_all['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Text Length (words) by Domain', fontsize=13, fontweight='bold')
domain_colors = {'news':'#4fc3f7','telegram':'#ef5350','review':'#66bb6a'}

for ax, (domain, df_d) in zip(axes, dfs.items()):
    df_d['text_len'] = df_d['text'].apply(lambda x: len(str(x).split()))
    ax.hist(df_d['text_len'].clip(0,150), bins=30,
            color=domain_colors[domain], alpha=0.85, edgecolor='white')
    ax.axvline(128, color='red', ls='--', lw=1.5, label='Max tokens (128)')
    ax.set_title(domain_names[domain], fontweight='bold')
    ax.set_xlabel('Word count'); ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
    ax.text(0.98,0.95,f'Median: {df_d["text_len"].median():.0f}w',
            transform=ax.transAxes, ha='right', va='top', fontsize=9)

plt.tight_layout()
plt.savefig('../results/text_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key: Telegram posts are shortest (avg ~18 words), news articles longest (avg ~62 words).')

## 4. Train / Val / Test Split

In [ ]:
LABEL2ID = {'negative':0,'neutral':1,'positive':2}
df_all['strat_key'] = df_all['label'] + '_' + df_all['domain']

train_df, temp_df = train_test_split(df_all, test_size=0.30,
                                     stratify=df_all['strat_key'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50,
                                     stratify=temp_df['strat_key'], random_state=42)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

os.makedirs('../data/processed', exist_ok=True)
for split, df in [('train',train_df),('val',val_df),('test',test_df)]:
    df[['text','label','domain']].to_csv(f'../data/processed/{split}.csv', index=False)

print('Saved to data/processed/')
print('Next: Notebook 02 — Lexicon + TF-IDF baselines')